In [2]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from langchain_community.vectorstores import FAISS
from typing import List
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain_openai import OpenAI
from langchain_community.document_loaders import PyMuPDFLoader

In [3]:
def find_pdfs(root_dir: str) -> List[Path]:
    root = Path(root_dir)
    # The rglob pattern is case-sensitive by default; to catch .PDF too, you could check suffix lowercased:
    pdfs = [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() == '.pdf']
    return pdfs


t = []
files = find_pdfs('./data/')

for i in files:
    FILE_PATH = str(i)  

    doc = PyMuPDFLoader(FILE_PATH, mode='page') # open a document
    text = doc.load()

    for i in range(len(text)):
        text[i].metadata['page'] += 1
    t.extend(text)

load_dotenv()

print(len(t))

text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=2000,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

texts = text_splitter.split_documents(t)

api_key= os.getenv("OPEN_AI_KEY")

TOP_K = 5

156


In [ ]:
EMBED_MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"

embedding = HuggingFaceEmbeddings(model_name=EMBED_MODEL_ID, model_kwargs={'device': 'cpu'})

vector_store = FAISS.from_documents(t, embedding)

In [6]:
#vector_store.save_local("faiss_index")
vector_store1 = FAISS.load_local("faiss_index", embedding, allow_dangerous_deserialization=True)

In [8]:
retriever = vector_store1.as_retriever(search_kwargs={"k": TOP_K})
llm = OpenAI(api_key=api_key)

In [9]:
keyword_retriever = BM25Retriever.from_documents(t)
keyword_retriever.k =  5
ensemble_retriever = EnsembleRetriever(retrievers=[retriever,keyword_retriever],weights=[0.5, 0.5])

In [14]:
user_prompt = "What dataset was Docling benchmarked on?"

retr = ensemble_retriever.get_relevant_documents(user_prompt)

texts= [(d.page_content, d.metadata) for d in retr]

print(texts[0])

('Pdf Parse\nOCR\nLayout\nTable Structure\nPage Total\n10\n2\n10\n1\n100\n101\nsec/page\nPdf Parse\nOCR\nLayout\nTable Structure\nPage Total\n0.0\n0.5\n1.0\n1.5\n2.0\n2.5\n3.0\nsec/page\nx86 CPU\nM3 Max\nL4 GPU\nFigure 4: Contributions of PDF backend and AI models to the conversion time of a page (in seconds per page). Lower is better.\nLeft: Ranges of time contributions for each model to pages it was applied on (i.e., OCR was applied only on pages with bitmaps,\ntable structure was applied only on pages with tables). Right: Average time contribution to a page in the benchmark dataset\n(factoring in zero-time contribution for OCR and table structure models on pages without bitmaps or tables) .\nDocling\nMarker\nMineru\nUnstructured\n0\n2\n4\n6\n8\n10\n12\n14\n16\nsec/page\nx86 CPU\nM3 Max\nL4 GPU\nFigure 5: Conversion time in seconds per page on our dataset\nin three scenarios, across all assets and system configura-\ntions. Lower bars are better. The configuration includes OCR\nand ta

In [ ]:
#Set up filter of metadata here

def slim_metadata(documents):
    """
    documents: list of tuples (text, metadata_dict)
    returns: new list of tuples (text, {'source': ..., 'page': ...})
    """
    slimmed = []
    for text, meta in documents:
        # pull only the two keys you care about
        src  = meta.get('file_path')  # or meta.get('source') if you renamed it earlier
        page = meta.get('page')
        slimmed.append((text, {'source': src, 'page': page}))
    return slimmed

slimmed_docs = [
    (text, {'source': m['file_path'], 'page': m['page']})
    for text, m in texts
]

print(slimmed_docs)


[('Pdf Parse\nOCR\nLayout\nTable Structure\nPage Total\n10\n2\n10\n1\n100\n101\nsec/page\nPdf Parse\nOCR\nLayout\nTable Structure\nPage Total\n0.0\n0.5\n1.0\n1.5\n2.0\n2.5\n3.0\nsec/page\nx86 CPU\nM3 Max\nL4 GPU\nFigure 4: Contributions of PDF backend and AI models to the conversion time of a page (in seconds per page). Lower is better.\nLeft: Ranges of time contributions for each model to pages it was applied on (i.e., OCR was applied only on pages with bitmaps,\ntable structure was applied only on pages with tables). Right: Average time contribution to a page in the benchmark dataset\n(factoring in zero-time contribution for OCR and table structure models on pages without bitmaps or tables) .\nDocling\nMarker\nMineru\nUnstructured\n0\n2\n4\n6\n8\n10\n12\n14\n16\nsec/page\nx86 CPU\nM3 Max\nL4 GPU\nFigure 5: Conversion time in seconds per page on our dataset\nin three scenarios, across all assets and system configura-\ntions. Lower bars are better. The configuration includes OCR\nand t

In [ ]:
llm = OpenAI(api_key=api_key, model='gpt-4o-mini')

prompt = PromptTemplate.from_template("You are currently a part of a RAG. Using these files: {input}, answer the user's prompt: {prompt} with the at most precesion possible. The files you have received are always organized with the page content first and then then metadata, so be sure to associate the right documents with the right metadata. Keep your answers clear and with as little filler as possible. Be to the point! If you believe that the piece of information within the prompt is not found in the given document chunks, say so rather than making up information. At the begining of your answer, cite the prompt given to you. At the end of your answer, give the source of the document your found the information in as well as the page number. If you found information from multiple pages or multpible document, cite them all.")

chain = prompt | llm
print(chain.invoke(
    {
        "prompt": user_prompt,
        "input": slimmed_docs
    }
))

 

What dataset was Docling benchmarked on?

Docling was benchmarked on a test set of 89 PDF files covering a variety of styles, features, content, and length, which included a total of 4008 pages, 56246 text items, 1842 tables, and 4676 pictures. The dataset was based on the DocLayNet dataset and augmented with additional samples from CCpdf.
